# Laptop Price Predictor

This project implements an end-to-end machine learning regression pipeline designed to predict laptop prices based on technical specifications and hardware features. Using a dataset of 1,275 laptops, the pipeline engineers key characteristics from unstructured fields to deliver accurate predictions.

### Key Stages:
- **Feature Engineering**: Parsed screen configurations to extract properties like touchscreen capability, IPS status, and Pixels Per Inch (PPI). Categorized complex CPU model strings into clean series and split memory specifications into primary/secondary capacities and storage types (e.g., SSD, HDD, Flash).
- **Preprocessing & Pipeline**: Built a robust `ColumnTransformer` to scale numerical features and one-hot encode categorical features, preventing data leakage during training.
- **Model Training & Evaluation**: Evaluated Linear Regression, Ridge, and Random Forest models using 5-fold cross-validation. The Random Forest model achieved the best performance with a cross-validated $R^2$ of ~0.88 and a root mean squared error of ~€280 on the test set.
- **Visualization**: Plotted the price distribution, feature importances, actual vs. predicted values, and residual analysis to confirm the model's performance and investigate key pricing drivers.

In [38]:
%pip install scikit-learn
%pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [39]:
import warnings
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.gridspec as gridspec
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_score

warnings.filterwarnings('ignore')

In [40]:
df = pd.read_csv("laptop_price - dataset.csv", encoding="latin")
print(f"Dataset: {len(df):,} laptops × {df.shape[1]} columns\n")
print(df[["Company","TypeName","ScreenResolution","CPU_Type",
          "RAM (GB)","Memory","GPU_Type","Price (Euro)"]].head(4).to_string())
df.head()

Dataset: 1,275 laptops × 15 columns

  Company   TypeName                    ScreenResolution       CPU_Type  RAM (GB)               Memory                GPU_Type  Price (Euro)
0   Apple  Ultrabook  IPS Panel Retina Display 2560x1600        Core i5         8            128GB SSD  Iris Plus Graphics 640       1339.69
1   Apple  Ultrabook                            1440x900        Core i5         8  128GB Flash Storage        HD Graphics 6000        898.94
2      HP   Notebook                   Full HD 1920x1080  Core i5 7200U         8            256GB SSD         HD Graphics 620        575.00
3   Apple  Ultrabook  IPS Panel Retina Display 2880x1800        Core i7        16            512GB SSD          Radeon Pro 455       2537.45


,Company,Product,TypeName,Inches,ScreenResolution,CPU_Company,CPU_Type,CPU_Frequency (GHz),RAM (GB),Memory,GPU_Company,GPU_Type,OpSys,Weight (kg),Price (Euro)
0,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,2.3,8,128GB SSD,Intel,Iris Plus Graphics 640,macOS,1.37,1339.69
1,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel,Core i5,1.8,8,128GB Flash Storage,Intel,HD Graphics 6000,macOS,1.34,898.94
2,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel,Core i5 7200U,2.5,8,256GB SSD,Intel,HD Graphics 620,No OS,1.86,575.00
3,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel,Core i7,2.7,16,512GB SSD,AMD,Radeon Pro 455,macOS,1.83,2537.45
4,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,3.1,8,256GB SSD,Intel,Iris Plus Graphics 650,macOS,1.37,1803.60


In [41]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 0 to 1274
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Company              1275 non-null   str    
 1   Product              1275 non-null   str    
 2   TypeName             1275 non-null   str    
 3   Inches               1275 non-null   float64
 4   ScreenResolution     1275 non-null   str    
 5   CPU_Company          1275 non-null   str    
 6   CPU_Type             1275 non-null   str    
 7   CPU_Frequency (GHz)  1275 non-null   float64
 8   RAM (GB)             1275 non-null   int64  
 9   Memory               1275 non-null   str    
 10  GPU_Company          1275 non-null   str    
 11  GPU_Type             1275 non-null   str    
 12  OpSys                1275 non-null   str    
 13  Weight (kg)          1275 non-null   float64
 14  Price (Euro)         1275 non-null   float64
dtypes: float64(4), int64(1), str(10)
memory usage: 14

Company                0
Product                0
TypeName               0
Inches                 0
ScreenResolution       0
CPU_Company            0
CPU_Type               0
CPU_Frequency (GHz)    0
RAM (GB)               0
Memory                 0
GPU_Company            0
GPU_Type               0
OpSys                  0
Weight (kg)            0
Price (Euro)           0
dtype: int64

### Feature Engineering: Screen Resolution
We parse `ScreenResolution` to extract:
- `is_touchscreen`
- `is_IPS`
- `res_width` & `res_height`
- `PPI` (Pixels Per Inch)

In [42]:
def parse_screen(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    sr = df["ScreenResolution"]
    df["Is_Touchscreen"] = sr.str.contains("Touchscreen", na=False).astype(int)
    df["Is_IPS"]         = sr.str.contains("IPS",         na=False).astype(int)
    # Extract e.g. "2560x1600" → width=2560, height=1600
    res = sr.str.extract(r"(\d{3,4})[xX](\d{3,4})")
    df["Res_Width"]  = res[0].astype(float).fillna(1366)
    df["Res_Height"] = res[1].astype(float).fillna(768)
    # Pixels Per Inch = diagonal_pixels / screen_inches
    df["PPI"] = (
        np.sqrt(df["Res_Width"] ** 2 + df["Res_Height"] ** 2) / df["Inches"]
    ).round(1)
    return df

### Feature Engineering: Memory
We split the `Memory` column to extract the size in GB of primary and secondary storage, and identify the storage types.

In [43]:

def _to_GB(val: str) -> int:
    M = re.search(r"(\d+\.?\d*)\s*(TB|GB)", str(val).upper())
    if not M:
        return 0
    n = float(M.group(1))
    unit = M.group(2)
    if unit == "TB":
        n *= 1024
    return int(n)


def _storage_type(s: str) -> str:
    s = s.upper()
    if "SSD" in s or "NVME" in s:    return "SSD"
    if "HDD" in s:                   return "HDD"
    if "FLASH" in s or "EMMC" in s:  return "Flash"
    return "Other"


def parse_storage(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Some entries have dual storage: "256GB SSD +  1TB HDD"
    primary   = df["Memory"].apply(lambda x: x.split("+")[0].strip())
    secondary = df["Memory"].apply(lambda x: x.split("+")[1].strip() if "+" in x else None)

    df["Primary_GB"]           = primary.apply(_to_GB)
    df["Has_Secondary"]        = secondary.notna().astype(int)
    df["Secondary_GB"]         = secondary.apply(lambda x: _to_GB(x) if x else 0)
    df["Total_Storage_GB"]     = df["Primary_GB"] + df["Secondary_GB"]
    df["Primary_Storage_Type"] = primary.apply(_storage_type)
    return df                    
                    



In [44]:
_CPU_MAP = [
    ("core i9",  "Core i9"),
    ("core i7",  "Core i7"),
    ("core i5",  "Core i5"),
    ("core i3",  "Core i3"),
    ("ryzen 7",  "Ryzen 7"),
    ("ryzen 5",  "Ryzen 5"),
    ("ryzen 3",  "Ryzen 3"),
    ("pentium",  "Pentium"),
    ("celeron",  "Celeron"),
    ("xeon",     "Xeon"),
    ("atom",     "Atom"),
]

def _cpu_series(text: str) -> str:
    t = text.lower()
    for keyword, label in _CPU_MAP:
        if keyword in t:
            return label
    return "AMD APU / Other"


def parse_cpu(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["CPU_Series"] = df["CPU_Type"].apply(_cpu_series)
    return df
    print(df)

for gpu


In [45]:
from pandas import unique
unique_gpus =df["GPU_Type"].unique()
unique_gpus = sorted(unique_gpus)
print(f"Total unique GPU types: {len(unique_gpus)}")
print(unique_gpus)


Total unique GPU types: 106
['FirePro W4190M', 'FirePro W5130M', 'FirePro W6150M', 'GTX 980 SLI', 'GeForce 150MX', 'GeForce 920', 'GeForce 920M', 'GeForce 920MX', 'GeForce 930M', 'GeForce 930MX', 'GeForce 940M', 'GeForce 940MX', 'GeForce 960M', 'GeForce GT 940MX', 'GeForce GTX 1050', 'GeForce GTX 1050 Ti', 'GeForce GTX 1050M', 'GeForce GTX 1050Ti', 'GeForce GTX 1060', 'GeForce GTX 1070', 'GeForce GTX 1070M', 'GeForce GTX 1080', 'GeForce GTX 930MX', 'GeForce GTX 940M', 'GeForce GTX 940MX', 'GeForce GTX 950M', 'GeForce GTX 960', 'GeForce GTX 960<U+039C>', 'GeForce GTX 960M', 'GeForce GTX 965M', 'GeForce GTX 970M', 'GeForce GTX 980', 'GeForce GTX 980M', 'GeForce GTX1050 Ti', 'GeForce GTX1060', 'GeForce GTX1080', 'GeForce MX130', 'GeForce MX150', 'Graphics 620', 'HD Graphics', 'HD Graphics 400', 'HD Graphics 405', 'HD Graphics 500', 'HD Graphics 505', 'HD Graphics 510', 'HD Graphics 515', 'HD Graphics 520', 'HD Graphics 530', 'HD Graphics 5300', 'HD Graphics 540', 'HD Graphics 6000', 'HD G

APPLY AL ENGINERRING STEPS


In [46]:
df = parse_screen(df)
df = parse_storage(df)
df = parse_cpu(df)

print("\n─── Engineered Features (first 5 rows) ───")
eng_cols = ["Is_Touchscreen","Is_IPS","PPI",
            "Primary_GB","Has_Secondary","Total_Storage_GB",
            "Primary_Storage_Type","CPU_Series"]
print(df[eng_cols].head(5).to_string())
print("\nCPU Series distribution:")
print(df["CPU_Series"].value_counts().to_string())
print("\nStorage type distribution:")
print(df["Primary_Storage_Type"].value_counts().to_string())



─── Engineered Features (first 5 rows) ───
   Is_Touchscreen  Is_IPS    PPI  Primary_GB  Has_Secondary  Total_Storage_GB Primary_Storage_Type CPU_Series
0               0       1  227.0         128              0               128                  SSD    Core i5
1               0       0  127.7         128              0               128                Flash    Core i5
2               0       0  141.2         256              0               256                  SSD    Core i5
3               0       1  220.5         512              0               512                  SSD    Core i7
4               0       1  227.0         256              0               256                  SSD    Core i5

CPU Series distribution:
CPU_Series
Core i7            515
Core i5            423
Core i3            134
AMD APU / Other     78
Celeron             78
Pentium             30
Atom                13
Xeon                 4

Storage type distribution:
Primary_Storage_Type
SSD      837
HDD      359


In [47]:
print (df.columns)

Index(['Company', 'Product', 'TypeName', 'Inches', 'ScreenResolution',
       'CPU_Company', 'CPU_Type', 'CPU_Frequency (GHz)', 'RAM (GB)', 'Memory',
       'GPU_Company', 'GPU_Type', 'OpSys', 'Weight (kg)', 'Price (Euro)',
       'Is_Touchscreen', 'Is_IPS', 'Res_Width', 'Res_Height', 'PPI',
       'Primary_GB', 'Has_Secondary', 'Secondary_GB', 'Total_Storage_GB',
       'Primary_Storage_Type', 'CPU_Series'],
      dtype='str')


DEFINE X,Y

In [48]:
NUM_COLS = [
    "Inches",
    "CPU_Frequency (GHz)",
    "RAM (GB)",
    "Weight (kg)",
    "PPI",
    "Is_Touchscreen",
    "Is_IPS",
    "Primary_GB",
    "Has_Secondary",
    "Total_Storage_GB",
]
CAT_COLS = [
    "Company",
    "TypeName",
    "CPU_Company",
    "CPU_Series",
    "GPU_Company",
    "OpSys",
    "Primary_Storage_Type",
]
x = df[NUM_COLS + CAT_COLS].copy()

In [49]:
y = np.log1p(df["Price (Euro)"])

PREPROCESSING PIPE LINE

In [50]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler() , NUM_COLS),
    ("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False), CAT_COLS),
])

cross validation


In [51]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge (α=10)":      Ridge(alpha=10),
    "Random Forest":     RandomForestRegressor(
                             n_estimators=200, max_features="sqrt",
                             random_state=42, n_jobs=-1),
}

cv_results = {}
print("\n─── 5-Fold Cross-Validation ───")
print(f"{'Model':<25}  {'R² mean':>9}  {'R² ±std':>8}  {'Log-RMSE':>10}")
print("─" * 60)

for name, model in models.items():
    pipe = Pipeline([("pre", preprocessor), ("model", model)])
    r2   = cross_val_score(pipe, x, y, cv=5, scoring="r2")
    mse  = cross_val_score(pipe, x, y, cv=5, scoring="neg_mean_squared_error")
    lrmse = np.sqrt(-mse)
    cv_results[name] = {
        "r2_mean":  r2.mean(),
        "r2_std":   r2.std(),
        "log_rmse": lrmse.mean(),
    }
    print(f"{name:<25}  {r2.mean():>9.4f}  {r2.std():>8.4f}  {lrmse.mean():>10.4f}")


─── 5-Fold Cross-Validation ───
Model                        R² mean   R² ±std    Log-RMSE
────────────────────────────────────────────────────────────
Linear Regression             0.8265    0.0371      0.2531
Ridge (α=10)                  0.8239    0.0374      0.2551
Random Forest                 0.8794    0.0205      0.2116


In [52]:




x_train , x_test , y_train , y_test = train_test_split(x ,y , test_size=0.2,random_state=42)

best_pipe = Pipeline([
    ("pre",   preprocessor),
    ("model", RandomForestRegressor(
                  n_estimators=200, max_features="sqrt",
                  random_state=42, n_jobs=-1)),
])
best_pipe.fit(x_train, y_train)

y_pred_log =best_pipe.predict(x_test)
y_pred_euro = np.expm1(y_pred_log)
y_true_euro = np.expm1(y_test)

test_r2 = r2_score(y_true_euro ,y_pred_euro)
test_rmse = np.sqrt(mean_squared_error(y_pred_euro,y_true_euro))
print(f"\n★ Random Forest (held-out test 20%): R² = {test_r2:.4f} | RMSE = €{test_rmse:.2f}")




★ Random Forest (held-out test 20%): R² = 0.8412 | RMSE = €280.77


FEATURE IMPORTANCE


In [53]:
from sklearn.utils.discovery import all_functions
cat_feat_names = list(
    best_pipe.named_steps["pre"]
    .named_transformers_["cat"]
    .get_feature_names_out(CAT_COLS)
)

all_feat_names = NUM_COLS + cat_feat_names
feat_imp = pd.Series(
    best_pipe.named_steps["model"].feature_importances_,
    index=all_feat_names,
).sort_values(ascending=False)

def _short_name(name:str) -> str:
    for col in sorted(CAT_COLS, key=len, reverse=True):
        if name.startswith(col + "_"):
            return name[len(col)+1:]
            return name

top15 = feat_imp.head(15).copy()
top15.index = [_short_name(n) for n in top15.index]

print("\nTop 10 features:")
print(top15.head(10).round(4).to_string())            



Top 10 features:
NaN         0.1627
Notebook    0.0918
NaN         0.0792
Core i7     0.0785
NaN         0.0701
NaN         0.0561
SSD         0.0556
NaN         0.0465
NaN         0.0443
HDD         0.0347


VISUALIZATION


In [54]:
from matplotlib.pyplot import title
BG, PANEL          = "#0f1117", "#1a1d27"
A_TEAL, A_RED, A_Y = "#00d4aa", "#ff6b6b", "#ffd93d"
TEXT, GRID         = "#e0e0e0", "#2a2d3a"

def _style(ax,title=""):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors="black" ,labelsize=8)
    for sp in ax.spines.values():sp.set_color(GRID)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():lbl.set_color("Blue")
    if title:
        ax.set_title(title, color=TEXT, fontsize=11, fontweight="bold", pad=8)

fig = plt.figure(figsize=(18, 16))
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.40)


        




<Figure size 1800x1600 with 0 Axes>

In [55]:
ax = fig.add_subplot(gs[0, 0])
ax.hist(df["Price (Euro)"], bins=50, color=A_TEAL, edgecolor=BG, alpha=0.85)
med = df["Price (Euro)"].median()
ax.axvline(med, color=A_RED, lw=2, ls="--", label=f"Median €{med:.0f}")
ax.set_xlabel("Price (€)", color=TEXT, fontsize=9)
ax.set_ylabel("Count",      color=TEXT, fontsize=9)
ax.legend(fontsize=8, labelcolor=TEXT, facecolor=PANEL, edgecolor=GRID)
_style(ax, "Price Distribution")

In [56]:
ax = fig.add_subplot(gs[0, 2])
ram_med = df.groupby("RAM (GB)")["Price (Euro)"].median()
ax.plot(ram_med.index, ram_med.values,
        color=A_TEAL, marker="o", markersize=6, lw=2)
ax.fill_between(ram_med.index, ram_med.values, alpha=0.15, color=A_TEAL)
ax.set_xlabel("RAM (GB)",          color=TEXT, fontsize=9)
ax.set_ylabel("Median Price (€)",  color=TEXT, fontsize=9)
_style(ax, "RAM vs Median Price")

In [57]:
ax = fig.add_subplot(gs[1, 0])
mnames   = list(cv_results.keys())
r2_means = [cv_results[m]["r2_mean"] for m in mnames]
r2_stds  = [cv_results[m]["r2_std"]  for m in mnames]
bars = ax.barh(
    mnames, r2_means, xerr=r2_stds,
    color=[A_TEAL, A_Y, A_RED], edgecolor=BG, height=0.5,
    error_kw=dict(ecolor=TEXT, lw=1.2, capsize=4),
)
ax.set_xlabel("R² Score", color=TEXT, fontsize=9)
ax.set_xlim(0, 1.08)
for bar, val in zip(bars, r2_means):
    ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", color=TEXT, va="center", fontsize=9, fontweight="bold")
_style(ax, "5-Fold CV R² Comparison")


In [58]:
ax = fig.add_subplot(gs[1, 1])
ax.scatter(y_true_euro, y_pred_euro, alpha=0.35, s=18,
           color=A_TEAL, edgecolors="none")
lim = max(float(y_true_euro.max()), float(y_pred_euro.max())) * 1.05
ax.plot([0, lim], [0, lim], color=A_RED, lw=1.5, ls="--", label="Perfect fit")
ax.set_xlabel("Actual Price (€)",    color=TEXT, fontsize=9)
ax.set_ylabel("Predicted Price (€)", color=TEXT, fontsize=9)
ax.legend(fontsize=8, labelcolor=TEXT, facecolor=PANEL, edgecolor=GRID)
ax.text(0.04, 0.88,
        f"R²   = {test_r2:.4f}\nRMSE = €{test_rmse:.0f}",
        transform=ax.transAxes, color=TEXT, fontsize=9,
        bbox=dict(facecolor=PANEL, edgecolor=GRID, boxstyle="round,pad=0.4"))
_style(ax, "Actual vs Predicted (Random Forest)")

In [59]:
ax = fig.add_subplot(gs[1, 2])
residuals = y_true_euro.values - y_pred_euro
ax.scatter(y_pred_euro, residuals, alpha=0.35, s=18,
           color=A_Y, edgecolors="none")
ax.axhline(0, color=A_RED, lw=1.5, ls="--")
ax.set_xlabel("Predicted (€)", color=TEXT, fontsize=9)
ax.set_ylabel("Residual (€)",  color=TEXT, fontsize=9)
_style(ax, "Residuals (No Systematic Bias = Good)")

In [60]:
# Ensure feature names are strings
top15.index = [str(n) for n in top15.index]

# Convert to plain lists for Matplotlib
labels = list(top15.index[::-1])   # feature names (strings)
values = list(top15.values[::-1])  # importance scores (floats)

# Plot
ax = fig.add_subplot(gs[2, :2])
fi_colors = [A_TEAL if i < 5 else "#4a7c8e" for i in range(len(top15))]

ax.barh(
    labels,
    values,
    color=fi_colors[::-1],
    edgecolor=BG,
    height=0.70
)

ax.set_xlabel("Feature Importance", color=TEXT, fontsize=9)
_style(ax, "Top 10 Feature Importances (Random Forest)")


In [61]:
import os

# Ensure the folder exists
os.makedirs("/mnt/user-data/outputs", exist_ok=True)

plt.savefig(
    "/mnt/user-data/outputs/laptop_price_predictor.png",
    dpi=150, bbox_inches="tight", facecolor=BG,
)


<Figure size 640x480 with 0 Axes>

In [62]:
plt.savefig("laptop_price_predictor.png", dpi=150, bbox_inches="tight", facecolor=BG)


<Figure size 640x480 with 0 Axes>

In [63]:
ax = fig.add_subplot(gs[2, 2])
cpu_med = df.groupby("CPU_Series")["Price (Euro)"].median().sort_values()
ax.barh(cpu_med.index, cpu_med.values, color=A_TEAL, edgecolor=BG, height=0.7)
ax.set_xlabel("Median Price (€)", color=TEXT, fontsize=9)
_style(ax, "Median Price by CPU Series")

fig.suptitle(
    "Laptop Price Predictor  ·  End-to-End ML Regression",
    color=TEXT, fontsize=16, fontweight="bold", y=0.995,
)
plt.savefig(
    "/mnt/user-data/outputs/laptop_price_predictor.png",
    dpi=150, bbox_inches="tight", facecolor=BG,
)
print("\nPlot saved → /mnt/user-data/outputs/laptop_price_predictor.png")


Plot saved → /mnt/user-data/outputs/laptop_price_predictor.png


<Figure size 640x480 with 0 Axes>